In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create DataLoader

In [2]:
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


# Обучаем Учителя и Студента для дальнейших дисциляций.

In [27]:
class Teacher(nn.Module):
    def __init__(self, num_classes=10):
        super(Teacher, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

class Student(nn.Module):
    def __init__(self, num_classes=10):
        super(Student, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [28]:
def train(model, train_loader, epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

def test(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [29]:
torch.manual_seed(42)

teacher = Teacher(num_classes=10).to(device)
student = Student(num_classes=10).to(device)
student_2 = Student(num_classes=10).to(device)

In [30]:
train(teacher, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_teacher = test(teacher, test_loader, device)

train(student, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_student = test(student, test_loader, device)

Epoch 1/10, Loss: 1.3995323801589439
Epoch 2/10, Loss: 0.9399766220765955
Epoch 3/10, Loss: 0.7362965358340222
Epoch 4/10, Loss: 0.5971776031319748
Epoch 5/10, Loss: 0.4753863841981229
Epoch 6/10, Loss: 0.36895610609322865
Epoch 7/10, Loss: 0.2752771711410464
Epoch 8/10, Loss: 0.2208312237087418
Epoch 9/10, Loss: 0.16930403992952897
Epoch 10/10, Loss: 0.1460103944915792
Test Accuracy: 73.85%
Epoch 1/10, Loss: 1.4786695144365511
Epoch 2/10, Loss: 1.1529739758242732
Epoch 3/10, Loss: 1.0139792451773153
Epoch 4/10, Loss: 0.9180379996214376
Epoch 5/10, Loss: 0.8398868606218597
Epoch 6/10, Loss: 0.7796025199963309
Epoch 7/10, Loss: 0.7175256077895689
Epoch 8/10, Loss: 0.6583287481914091
Epoch 9/10, Loss: 0.6122005651978886
Epoch 10/10, Loss: 0.5602635522480206
Test Accuracy: 70.80%


In [7]:
print(f"Teacher accuracy: {test_accuracy_teacher:.2f}%")
print(f"Student accuracy: {test_accuracy_student:.2f}%")

Teacher accuracy: 73.18%
Student accuracy: 69.80%


# Эксперимент 1 - Дистилляция логитов

In [8]:
def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.eval() 
    student.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)

            soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size()[0] * (T**2)
            label_loss = ce_loss(student_logits, labels)
            loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

In [9]:
train_knowledge_distillation(teacher=teacher, student=student_2, train_loader=train_loader, epochs=10, learning_rate=0.001, T=2, soft_target_loss_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_student_with_teacher_logit = test(student_2, test_loader, device)

Epoch 1/10, Loss: 2.4065874162537364
Epoch 2/10, Loss: 1.8444213534864928
Epoch 3/10, Loss: 1.5858703220591825
Epoch 4/10, Loss: 1.4328441900365494
Epoch 5/10, Loss: 1.3016334778207648
Epoch 6/10, Loss: 1.1967345473101683
Epoch 7/10, Loss: 1.117605957991022
Epoch 8/10, Loss: 1.0308501130479681
Epoch 9/10, Loss: 0.9604321100827679
Epoch 10/10, Loss: 0.8960097903180915
Test Accuracy: 70.84%


In [10]:
print(f"Teacher accuracy: {test_accuracy_teacher:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_student:.2f}%")
print(f"Student accuracy with Teacher: {test_accuracy_student_with_teacher_logit:.2f}%")

Teacher accuracy: 73.18%
Student accuracy without teacher: 69.80%
Student accuracy with Teacher: 70.84%


# Эксперимент 2 - Учим Студента совпадать по скрытому состоянию с Учителем (без модификации и обучения архитектур)

In [11]:
class ModifiedTeacher(nn.Module):
    def __init__(self, num_classes=10):
        super(ModifiedTeacher, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        flattened_conv_output = torch.flatten(x, 1)
        x = self.classifier(flattened_conv_output)
        flattened_conv_output_after_pooling = torch.nn.functional.avg_pool1d(flattened_conv_output, 2)
        return x, flattened_conv_output_after_pooling

class ModifiedStudent(nn.Module):
    def __init__(self, num_classes=10):
        super(ModifiedStudent, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        flattened_conv_output = torch.flatten(x, 1)
        x = self.classifier(flattened_conv_output)
        return x, flattened_conv_output

In [12]:
torch.manual_seed(42)

modified_teacher = ModifiedTeacher(num_classes=10).to(device)
modified_teacher.load_state_dict(teacher.state_dict())

modified_student = ModifiedStudent(num_classes=10).to(device)

In [13]:
sample = torch.randn(128, 3, 32, 32).to(device) 

logits, hidden_representation = modified_student(sample)
logits, hidden_representation = modified_teacher(sample)


print("Student logits shape:", logits.shape)
print("Student hidden representation shape:", hidden_representation.shape)
print()
print("Teacher logits shape:", logits.shape)
print("Teacher hidden representation shape:", hidden_representation.shape)

Student logits shape: torch.Size([128, 10])
Student hidden representation shape: torch.Size([128, 1024])

Teacher logits shape: torch.Size([128, 10])
Teacher hidden representation shape: torch.Size([128, 1024])


In [14]:
def train_cosine_loss(teacher, student, train_loader, epochs, learning_rate, hidden_rep_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.to(device)
    student.to(device)
    teacher.eval() 
    student.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            with torch.no_grad():
                _, teacher_hidden_representation = teacher(inputs)

            student_logits, student_hidden_representation = student(inputs)
            hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation, target=torch.ones(inputs.size(0)).to(device))

            label_loss = ce_loss(student_logits, labels)
            loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

def test_modified(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs, _ = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [15]:
train_cosine_loss(teacher=modified_teacher, student=modified_student, train_loader=train_loader, epochs=10, learning_rate=0.001, hidden_rep_loss_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_student_with_teacher_cosine = test_modified(modified_student, test_loader, device)

Epoch 1/10, Loss: 1.3089290445722888
Epoch 2/10, Loss: 1.064919425855817
Epoch 3/10, Loss: 0.9599955802988214
Epoch 4/10, Loss: 0.8792141956441543
Epoch 5/10, Loss: 0.825023900974742
Epoch 6/10, Loss: 0.7784146902811192
Epoch 7/10, Loss: 0.7335433752640433
Epoch 8/10, Loss: 0.7015394475453954
Epoch 9/10, Loss: 0.6666287221109776
Epoch 10/10, Loss: 0.6345762249911228
Test Accuracy: 70.31%


# Эксперимент 3 - Добавляем обучаемый регрессор

In [16]:
sample = torch.randn(128, 3, 32, 32).to(device) 

conv_fe_output_student = student.features(sample)
conv_fe_output_teacher = teacher.features(sample)

print("Student feature extractor: ", conv_fe_output_student.shape)
print("Teacher feature extractor: ", conv_fe_output_teacher.shape)

Student feature extractor:  torch.Size([128, 16, 8, 8])
Teacher feature extractor:  torch.Size([128, 32, 8, 8])


In [17]:
class ModifiedTeacherRegressor(nn.Module):
    def __init__(self, num_classes=10):
        super(ModifiedTeacherRegressor, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        conv_feature_map = x
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x, conv_feature_map

class ModifiedStudentRegressor(nn.Module):
    def __init__(self, num_classes=10):
        super(ModifiedStudentRegressor, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.regressor = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        regressor_output = self.regressor(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x, regressor_output

In [18]:
def train_mse_loss(teacher, student, train_loader, epochs, learning_rate, feature_map_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.to(device)
    student.to(device)
    teacher.eval()  
    student.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                _, teacher_feature_map = teacher(inputs)

            student_logits, regressor_feature_map = student(inputs)
            hidden_rep_loss = mse_loss(regressor_feature_map, teacher_feature_map)

            label_loss = ce_loss(student_logits, labels)
            loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

In [19]:
torch.manual_seed(42)
modified_student_reg = ModifiedStudentRegressor(num_classes=10).to(device)

modified_teacher_reg = ModifiedTeacherRegressor(num_classes=10).to(device)
modified_teacher_reg.load_state_dict(teacher.state_dict())

train_mse_loss(teacher=modified_teacher_reg, student=modified_student_reg, train_loader=train_loader, epochs=10, learning_rate=0.001, feature_map_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_student_with_teacher_reg = test_modified(modified_student_reg, test_loader, device)

Epoch 1/10, Loss: 1.7026713345666675
Epoch 2/10, Loss: 1.3320428297647735
Epoch 3/10, Loss: 1.1920501643129626
Epoch 4/10, Loss: 1.0924599204221954
Epoch 5/10, Loss: 1.013521563976317
Epoch 6/10, Loss: 0.9487619517404405
Epoch 7/10, Loss: 0.894032248145784
Epoch 8/10, Loss: 0.8463394486385843
Epoch 9/10, Loss: 0.8037383431363898
Epoch 10/10, Loss: 0.7675956202589947
Test Accuracy: 70.95%


In [20]:
print(f"Teacher accuracy: {test_accuracy_teacher:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_student:.2f}%")
print(f"Student accuracy with Teacher + Logit: {test_accuracy_student_with_teacher_logit:.2f}%")
print(f"Student accuracy with Teacher + Cosine: {test_accuracy_student_with_teacher_cosine:.2f}%")
print(f"Student accuracy with Teacher + Regressor: {test_accuracy_student_with_teacher_reg:.2f}%")

Teacher accuracy: 73.18%
Student accuracy without teacher: 69.80%
Student accuracy with Teacher + Logit: 70.84%
Student accuracy with Teacher + Cosine: 70.31%
Student accuracy with Teacher + Regressor: 70.95%


# Новый Эксперимент - Комбенируем несколько дисциляций

In [49]:
class CombineTeacher(nn.Module):
    def __init__(self, num_classes=10):
        super(CombineTeacher, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        flattened_conv_output = torch.flatten(x, 1)
        x = self.classifier(flattened_conv_output)
        flattened_conv_output_after_pooling = torch.nn.functional.avg_pool1d(flattened_conv_output, 2)
        return x, flattened_conv_output_after_pooling


class CombineStudent(nn.Module):
    def __init__(self, num_classes=10):
        super(CombineStudent, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        flattened_conv_output = torch.flatten(x, 1)
        x = self.classifier(flattened_conv_output)
        return x, flattened_conv_output


def test_combine(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs, _ = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [44]:
combine_teacher = CombineTeacher(num_classes=10).to(device)
combine_teacher.load_state_dict(teacher.state_dict())

combine_student = CombineStudent(num_classes=10).to(device)
combine_student.load_state_dict(student.state_dict())

<All keys matched successfully>

In [45]:
def train_combined_distillation(
    teacher, 
    student, 
    train_loader, 
    epochs, 
    learning_rate, 
    T, 
    soft_target_loss_weight, 
    hidden_rep_loss_weight, 
    ce_loss_weight, 
    device
):
    ce_loss = nn.CrossEntropyLoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.to(device)
    student.to(device)
    teacher.eval() 
    student.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            with torch.no_grad():
                teacher_logits, teacher_hidden_representation = teacher(inputs)

            student_logits, student_hidden_representation = student(inputs)
            hidden_rep_loss = cosine_loss(
                student_hidden_representation,
                teacher_hidden_representation,
                target=torch.ones(inputs.size(0)).to(device)
            )

            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)
            soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size(0) * (T**2)

            label_loss = ce_loss(student_logits, labels)
            loss = (
                hidden_rep_loss_weight * hidden_rep_loss +
                soft_target_loss_weight * soft_targets_loss +
                ce_loss_weight * label_loss
            )

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")


In [47]:
train_combined_distillation(teacher=combine_teacher, student=combine_student, train_loader=train_loader, epochs=10, learning_rate=0.001, T=2, soft_target_loss_weight=0.25, hidden_rep_loss_weight=0.25, ce_loss_weight=0.75, device=device)

Epoch 1/10, Loss: 1.004766015445485
Epoch 2/10, Loss: 0.9450658393637908
Epoch 3/10, Loss: 0.8905947912684486
Epoch 4/10, Loss: 0.8436713386374666
Epoch 5/10, Loss: 0.8036285983327099
Epoch 6/10, Loss: 0.7651992284733317
Epoch 7/10, Loss: 0.7319867917338906
Epoch 8/10, Loss: 0.7007100530292677
Epoch 9/10, Loss: 0.6731728016567962
Epoch 10/10, Loss: 0.6484075753432711


In [53]:
accuracy_teacher_combine = test_combine(combine_teacher, test_loader, device)
accuracy_student_combine = test_combine(combine_student, test_loader, device)

Test Accuracy: 73.85%
Test Accuracy: 71.15%


In [55]:
print(f"Teacher accuracy: {accuracy_teacher_combine:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_student:.2f}%")
print(f"Student accuracy with Teacher: {accuracy_student_combine:.2f}%")

Teacher accuracy: 73.85%
Student accuracy without teacher: 70.80%
Student accuracy with Teacher: 71.15%
